In [1]:
%matplotlib inline

In [ ]:
import argparse
import math
import os
import random
import subprocess
import sys
import time
from copy import deepcopy
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt

import snntorch as snn
from snntorch import utils

try:
    import comet_ml  # must be imported before torch (if installed)
except ImportError:
    comet_ml = None

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import yaml
from torch.optim import lr_scheduler
from tqdm import tqdm

pat
FILE = Path(pat).resolve()
ROOT = FILE.parents[0]  # YOLOv3 root directory
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))  # add ROOT to PATH
ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative

import val as validate  # for end-of-epoch mAP
from models.experimental import attempt_load
# from models.yolo import Model

try:
    import thop  # for FLOPs computation
except ImportError:
    thop = None
from utils.plots import feature_visualization

from utils.autoanchor import check_anchors, check_anchor_order
from utils.autobatch import check_train_batch_size
from utils.callbacks import Callbacks
from utils.dataloaders import create_dataloader
from utils.downloads import attempt_download, is_url
from utils.general import (
    LOGGER,
    TQDM_BAR_FORMAT,
    check_amp,
    check_dataset,
    check_file,
    check_git_info,
    check_git_status,
    check_img_size,
    check_requirements,
    check_suffix,
    check_yaml,
    colorstr,
    get_latest_run,
    increment_path,
    init_seeds,
    intersect_dicts,
    labels_to_class_weights,
    labels_to_image_weights,
    methods,
    one_cycle,
    print_args,
    print_mutation,
    strip_optimizer,
    yaml_save,
)
from utils.loggers import Loggers
from utils.loggers.comet.comet_utils import check_comet_resume
from utils.loss import ComputeLoss
from utils.metrics import fitness
from utils.torch_utils import (
    EarlyStopping,
    ModelEMA,
    initialize_weights,
    de_parallel,
    select_device,
    smart_DDP,
    smart_optimizer,
    smart_resume,
    torch_distributed_zero_first,
    time_sync,
    fuse_conv_and_bn,
    model_info,
)
from mymodels import *

LOCAL_RANK = int(os.getenv("LOCAL_RANK", -1))  # https://pytorch.org/docs/stable/elastic/run.html
RANK = int(os.getenv("RANK", -1))
WORLD_SIZE = int(os.getenv("WORLD_SIZE", 1))
GIT_INFO = check_git_info()
# print(f"LOCAL_RANK: {LOCAL_RANK}, RANK: {RANK}, WORLD_SIZE: {WORLD_SIZE}")
def parse_opt(known=False):
    """
    Parse command line arguments for configuring the training of a YOLO model.

    Args:
        known (bool): Flag to parse known arguments only, defaults to False.

    Returns:
        (argparse.Namespace): Parsed command line arguments.

    Examples:
        ```python
        options = parse_opt()
        print(options.weights)
        ```

    Notes:
        * The default weights path is 'yolov3-tiny.pt'.
        * Set `known` to True for parsing only the known arguments, useful for partial arguments.

    References:
        * Models: https://github.com/ultralytics/yolov5/tree/master/models
        * Datasets: https://github.com/ultralytics/yolov5/tree/master/data
        * Training Tutorial: https://docs.ultralytics.com/yolov5/tutorials/train_custom_data
    """

    parser = argparse.Namespace()
    parser.weights = ''#ROOT / "yolov3-tiny.pt"
    parser.cfg = "models/CNNEMS.yaml"
    parser.data = "dataset.yaml"
    parser.hyp = ROOT / "data/hyps/hyp.scratch-low.yaml"
    parser.epochs = 120
    parser.batch_size = 32
    parser.imgsz = 200
    parser.rect = False
    parser.resume = False
    parser.nosave = False
    parser.noval = False
    parser.noautoanchor = False
    parser.noplots = False
    parser.evolve = None
    
    parser.bucket = ""
    parser.cache = None
    parser.image_weights = False
    parser.device = ""
    parser.multi_scale = False
    parser.single_cls = False
    parser.optimizer = "SGD"
    parser.sync_bn = False
    parser.workers = 8
    parser.project = ROOT / "runs/val"
    parser.name = "exp"
    parser.exist_ok = False
    parser.quad = False
    parser.cos_lr = False
    parser.label_smoothing = 0.0
    parser.patience = 100
    parser.freeze = [0]
    parser.save_period = -1
    parser.seed = 0
    parser.local_rank = -1

    parser.entity = None
    parser.upload_dataset = False
    parser.bbox_interval = -1
    parser.artifact_alias = "latest"
    parser.time_steps = 10
    parser.save_dir = str(increment_path(Path(parser.project) / parser.name, exist_ok=parser.exist_ok))

    return parser

In [ ]:
f = Path("/home/mati/magisterka/mySpikingYolo/runs/train/EMSSnnPiramide_learnable_threshold_learnable_beta_1") / "weights" / "best.pt"
opt = parse_opt()
(print(f))
if f.exists():
    device = torch.device("cuda:0")
    ckpt = torch.load(f, map_location=device)  # load
    for k in ckpt.keys():
        print(k)

    # model = EMSSnnPiramide(time_steps=3, membrane_decay=0.8, learn_threshold=True, learn_beta = False, grad_func = surrogate.fast_sigmoid())
    model = EMSSnnPiramide(time_steps=3, membrane_decay=0.8, learn_threshold=True, learn_beta=True, grad_func=None)
    # model = CNNEMS()
    
    sss = 256
    model.detect.stride = torch.tensor([sss / x.shape[-2] for x in model.forward(torch.zeros(1, 3, sss, sss))])# forward
    check_anchor_order(model.detect)
    model.detect.anchors /= model.detect.stride.view(-1, 1, 1)
    model.stride = model.detect.stride
    model.detect.stride = model.detect.stride.to(device)
    model.load_state_dict(ckpt["model_state_dict"]) 
    model.float() # load model
    callbacks = Callbacks()
    data_dict = check_dataset(opt.data)  # check
    gs = max(int(model.stride.max()), 32)
    imgsz = check_img_size(opt.imgsz, gs, floor=gs * 2)
    val_path = data_dict["val"]
    nl = model.detect.nl
    with open(opt.hyp) as f:
        hyp = yaml.safe_load(f)  # load hyps
    nc = 1 if opt.single_cls else int(data_dict["nc"])
    names = {0: "item"} if opt.single_cls and len(data_dict["names"]) != 1 else data_dict["names"]  # class names
    hyp["box"] *= 3 / nl  # scale to layers
    hyp["cls"] *= nc / 80 * 3 / nl  # scale to classes and layers
    hyp["obj"] *= (imgsz / 640) ** 2 * 3 / nl  # scale to image size and layers
    hyp["label_smoothing"] = opt.label_smoothing


    val_loader, dataset = create_dataloader(
            val_path,
            imgsz,
            opt.batch_size // WORLD_SIZE * 2,
            gs,
            opt.single_cls,
            hyp=hyp,
            cache=None if opt.noval else opt.cache,
            rect=True,
            rank=-1,
            workers=opt.workers * 2,
            pad=0.5,
            prefix=colorstr("val: "),
        )
    
    save_directory = Path(opt.save_dir)
    (save_directory.parent if opt.evolve else save_directory).mkdir(parents=True, exist_ok=True)
    model.nc = nc  # attach number of classes to model
    model.hyp = hyp  # attach hyperparameters to model
    model.class_weights = labels_to_class_weights(dataset.labels, nc).to(device) * nc  # attach class weights
    model.names = names
    model.to(device)

    results, _, _ = validate.run(
        data_dict,
        batch_size=opt.batch_size // WORLD_SIZE * 2,
        imgsz=imgsz,
        # model=attempt_load(f, device).half(),
        model = model,
        # model = model_eval,
        iou_thres=0.6,#0.65 if is_coco else 0.60,  # best pycocotools at iou 0.65
        single_cls=opt.single_cls,
        dataloader=val_loader,
        save_dir=save_directory,
        #save_json=is_coco,
        verbose=True,
        half=False,
        plots=True,
        # plots = False,
        callbacks=callbacks,
        compute_loss=ComputeLoss(model),
    )  # val best model with plots
    
    torch.cuda.empty_cache()

In [ ]:
"17-08-29_10-56-11_488500000_548500000_histogram_19.png"

In [15]:
(im, targets, paths, shapes) = next(iter(val_loader))

In [6]:
# def plot_images(images, targets, paths=None, fname="images.jpg", names=None):
#     """Plots a grid of images with labels, optionally resizing and annotating with target boxes and names."""
#     if isinstance(images, torch.Tensor):
#         images = images.cpu().float().numpy()
#     if isinstance(targets, torch.Tensor):
#         targets = targets.cpu().numpy()

#     max_size = 1920  # max image size
#     max_subplots = 16  # max image subplots, i.e. 4x4
#     bs, _, h, w = images.shape  # batch size, _, height, width
#     bs = min(bs, max_subplots)  # limit plot images
#     ns = np.ceil(bs**0.5)  # number of subplots (square)
#     if np.max(images[0]) <= 1:
#         images *= 255  # de-normalise (optional)

#     # Build Image
#     mosaic = np.full((int(ns * h), int(ns * w), 3), 255, dtype=np.uint8)  # init
#     for i, im in enumerate(images):
#         if i == max_subplots:  # if last batch has fewer images than we expect
#             break
#         x, y = int(w * (i // ns)), int(h * (i % ns))  # block origin
#         im = im.transpose(1, 2, 0)
#         mosaic[y : y + h, x : x + w, :] = im

#     # Resize (optional)
#     scale = max_size / ns / max(h, w)
#     if scale < 1:
#         h = math.ceil(scale * h)
#         w = math.ceil(scale * w)
#         mosaic = cv2.resize(mosaic, tuple(int(x * ns) for x in (w, h)))

#     # Annotate
#     fs = int((h + w) * ns * 0.01)  # font size
#     annotator = Annotator(mosaic, line_width=round(fs / 10), font_size=fs, pil=True, example=names)
#     for i in range(i + 1):
#         x, y = int(w * (i // ns)), int(h * (i % ns))  # block origin
#         annotator.rectangle([x, y, x + w, y + h], None, (255, 255, 255), width=2)  # borders
#         if paths:
#             annotator.text([x + 5, y + 5], text=Path(paths[i]).name[:40], txt_color=(220, 220, 220))  # filenames
#         if len(targets) > 0:
#             ti = targets[targets[:, 0] == i]  # image targets
#             boxes = xywh2xyxy(ti[:, 2:6]).T
#             classes = ti[:, 1].astype("int")
#             labels = ti.shape[1] == 6  # labels if no conf column
#             conf = None if labels else ti[:, 6]  # check for confidence presence (label vs pred)

#             if boxes.shape[1]:
#                 if boxes.max() <= 1.01:  # if normalized with tolerance 0.01
#                     boxes[[0, 2]] *= w  # scale to pixels
#                     boxes[[1, 3]] *= h
#                 elif scale < 1:  # absolute coords need scale if image scales
#                     boxes *= scale
#             boxes[[0, 2]] += x
#             boxes[[1, 3]] += y
#             for j, box in enumerate(boxes.T.tolist()):
#                 cls = classes[j]
#                 color = colors(cls)
#                 cls = names[cls] if names else cls
#                 if labels or conf[j] > 0.25:  # 0.25 conf thresh
#                     label = f"{cls}" if labels else f"{cls} {conf[j]:.1f}"
#                     annotator.box_label(box, label, color=color)
#     annotator.im.save(fname)  # save

In [16]:
def output_to_target(output, max_det=300):
    """Converts model output to [batch_id, class_id, x, y, w, h, conf] format for plotting, handling up to `max_det`
    detections.
    """
    targets = []
    for i, o in enumerate(output):
        box, conf, cls = o[:max_det, :6].cpu().split((4, 1, 1), 1)
        j = torch.full((conf.shape[0], 1), i)
        targets.append(torch.cat((j, cls, xyxy2xywh(box), conf), 1))
    return torch.cat(targets, 0).numpy()

In [17]:
conf_thres=0.001
iou_thres=0.6
lb=[]
(im, targets, paths, shapes) = next(iter(val_loader))

In [18]:
import time

In [19]:
im = im.float()  # uint8 to fp16/32
im /= 255  # 0 - 255 to 0.0 - 1.0
nb, _, height, width = im.shape  # batch size, channels, height, width
model.eval()
model.to(device)
im = im.to(device)



In [20]:
start = time.time()
with torch.no_grad():
    preds, train_out = model(im)  # forward
    preds = non_max_suppression(
                preds, conf_thres, iou_thres, labels=lb, multi_label=True, agnostic=False, max_det=300
            )
end = time.time()

In [ ]:
print((end-start)*1000)

In [13]:
# plot_images(im, targets, paths, names=names)
# plot_images(im, output_to_target(preds), paths, names=names, fname="preds.jpg")